# EE6037 - Core concepts of Agentic AI Lab

This lab is designed to cover some of the key concepts from the lecture:
Agents, context, RAG, Tools - these are core components of Agents that once familiar with, you should be able to help you with other Agentic competions on Kaggle. 

The lab is centred around a ficticious company "Ironwood Labs" - we control the data about this company and investigate how much it takes for agents to understand our company!

## Part 0 — Setup

Part 0 covers the dependencies and set up in order for this lab to run.

We use "Ollama" to host and run our LLM for this lab, and the agentic framework "Agno" to serve as our "harness".

In [2]:
# Install Ollama (the local LLM server)
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -5

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 0s (2,496 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 125659 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
# Start the Ollama server as a background process, then wait for it to be ready
import subprocess, time, requests

subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)

# Poll until the server responds
for i in range(30):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=1)
        print(f"✅ Ollama is up (took {i+1}s)")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start — check /tmp/ollama.log")

✅ Ollama is up (took 3s)


In [4]:
# Pull the model — this takes ~2 minutes the first time
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling dde5aa3fc5ff:   2% ▕                  ▏  34 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   4% ▕                  ▏  90 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   6% ▕█                 ▏ 121 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   9% ▕█                 ▏ 177 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  11% ▕██                ▏ 224 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  13% ▕██                ▏ 259 MB/2.0 GB                  pulling manifest 
pulling dde5

In [5]:
# Install the Python libraries we need
!pip install -q agno ollama lancedb sentence-transformers ddgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 733.7 kB/s eta 0:00:000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 44.9 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.2/75.2 MB 8.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 749.8 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 1.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.8/406.8 kB 10.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 50.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 1.7 MB/s eta 0:00:00eta 0:00:01


## Part 1 — What is an agent, really?

Before building the whole team, let's see the smallest possible thing that
still counts as an "agent" in Agno: an LLM wrapped in a loop, with an
identity (a `name`, `role`, `instructions`) and — soon — access to tools.

Right now, no tools. Just the LLM.

In [6]:
from agno.agent import Agent
from agno.models.ollama import Ollama

plain_agent = Agent(
    name="Plain Agent",

    model=Ollama(id="llama3.2"),
 
    instructions=[
        "You are a helpful assistant.",
        "If you don't know something, say so plainly. Do not make things up.",
    ],
    markdown=True,
    debug_mode=True,
)

In [7]:
plain_agent.print_response("What is Ironwood Labs known for?", stream=True)

# Interestingly across multiple runs the LLM gives different answers to this question, despite receiveing the same inputs each time,
# showcasing the non-deterministic nature of these models. The answers recieved included:
# *I don't know, which in theory is the correct answer for this made up company
# *Ironwood Pharmaceuticals, which seems to have possibly been named Ironwood Labs in the past
# *Complete hallucination, telling me Ironwood labs developed the games Rimworld and Faeria, two both of wich are completely incorrect

DEBUG Agent initialized: plain-agent

DEBUG Creating new AgentSession: 205d486a-5e87-4b6a-92ee-3bd8ee367a87

DEBUG Agent Run Start: c5cb5096-842c-4daa-a0f5-a9879192aae1

DEBUG Ollama Response Stream Start

DEBUG Model: llama3.2

DEBUG ========================== system ==========================

... (11 outputs left out)

DEBUG  METRICS

DEBUG Ollama Response Stream End

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: c5cb5096-842c-4daa-a0f5-a9879192aae1

## Questions on Simple Agent

**What to do:** use the code cell below to ask `plain_agent` **at least 3 questions** — copy the
`plain_agent.print_response(...)` line from above and change the question each time. Include:
- one general-knowledge question (e.g. *"What is the capital of France?"*)
- one up-to-date question (e.g. *"What is today's date?"* or *"Who won yesterday's match?"*)
- one question about the fictional company *"Ironwood Labs"*

Then answer the questions in the *Answers* cell.

> **How to answer:** questions marked ✍️ are **written** — double-click the *Answers* cell below and type your answer after the colon.
> Questions marked 💻 need **code** — copy the relevant cell from above into the empty code cell provided, edit it, and run it.
> Save your notebook (**File → Save** / Ctrl+S) when you are done with each part.

1. ✍️ Which types of question does the agent get correct? Why?
2. ✍️ Which types of question can the model **not** answer? What does it sometimes do when it doesn't know the answer, and why is that dangerous?
3. ✍️ How could we improve the model's responses to these questions?
4. ✍️ Can you identify the Sytem and user prompt in the above example?


In [8]:
# 💻 Ask plain_agent your questions here (copy the print_response line from above and change the question)
plain_agent.print_response("What was the main contributing reason to the fall of Napoleon III", stream=True)

DEBUG Session ID: 205d486a-5e87-4b6a-92ee-3bd8ee367a87

DEBUG Agent initialized: plain-agent

DEBUG Creating new AgentSession: 205d486a-5e87-4b6a-92ee-3bd8ee367a87

DEBUG Agent Run Start: 63f1b1e1-b4c7-4137-b1f5-a09c27411464

DEBUG Ollama Response Stream Start

DEBUG Model: llama3.2

... (12 outputs left out)

DEBUG  METRICS

DEBUG Ollama Response Stream End

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: 63f1b1e1-b4c7-4137-b1f5-a09c27411464

In one run, the model gave me the answer that the Paris Commune was the main cause of Napoleon III's fall, which is largely incorrect. It more correctly told me that the immediate cause was the Franco-Prussian in the majority of runs

In [9]:
# 💻 Ask plain_agent your questions here (copy the print_response line from above and change the question)
plain_agent.print_response("When was the most recent Irish soccer match, and what was the result", stream=True)

DEBUG Session ID: 205d486a-5e87-4b6a-92ee-3bd8ee367a87

DEBUG Agent initialized: plain-agent

DEBUG Creating new AgentSession: 205d486a-5e87-4b6a-92ee-3bd8ee367a87

DEBUG Agent Run Start: 081201cb-117e-4382-a212-32b66343d268

DEBUG Ollama Response Stream Start

DEBUG Model: llama3.2

... (12 outputs left out)

DEBUG  METRICS

DEBUG Ollama Response Stream End

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: 081201cb-117e-4382-a212-32b66343d268

In [10]:
# 💻 Ask plain_agent your questions here (copy the print_response line from above and change the question)
plain_agent.print_response("What valuation was placed on Ironwood Labs when it IPO'd in 2021?", stream=True)

DEBUG Session ID: 205d486a-5e87-4b6a-92ee-3bd8ee367a87

DEBUG Agent initialized: plain-agent

DEBUG Creating new AgentSession: 205d486a-5e87-4b6a-92ee-3bd8ee367a87

DEBUG Agent Run Start: 6ade7968-543e-41b0-ac45-296919dfe4a9

DEBUG Ollama Response Stream Start

DEBUG Model: llama3.2

... (12 outputs left out)

DEBUG  METRICS

DEBUG Ollama Response Stream End

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: 6ade7968-543e-41b0-ac45-296919dfe4a9

### Answers — Simple Agent
1. Which types of question does it get correct, and why: It usually gets questions relating to historical data correct, information that would be found in its training dataset, as it would have scene documents that answered these questions during its training.
2. Which types can it not answer / what does it do when it doesn't know / why is that dangerous:
* It cannot answer up to date questions, as it is not connected to the web, and so cannot access info from after 2023 as that was this particular models training-cutoff date.
* It may answer that it does not know, and looks for clarifications/suggests where the information can be found, or create a fake answer, depending on the circumstances
* If you are basing decisions on the hallucinated answers, these decsions are being made with incomplete or erroneus info
4. How could we improve its responses:
4a. System prompt: "If you don't know something, say so plainly. Do not make things up."
4b. User prompt: "please anwser this question with information you can support with references, do not create false information"

## Part 2 — Giving an agent tools

An agent with tools can decide, *at inference time*, to call a function to
fetch information it doesn't have, rather than hallucinate. The model sees tool descriptions in its
prompt and emits structured calls that Agno actually executes.

Let's give an agent access to the internet through WebSearchTools so it can fetch information that is not naturally in the models learned "memory".

Tools can be part of either a "toolkit" like the WebSearchTools tool we apply below, or they can simply be fully custom code (or if you want to be fancy - they can even be small cheap models for very specific tasks)

In [11]:
from agno.agent import Agent
from agno.models.ollama import Ollama
from agno.tools.websearch import WebSearchTools


scout = Agent(
    name="Scout",
    role="Search the internet for up to date information",
    
    model=Ollama(id="llama3.2"),

    tools=[WebSearchTools(backend="duckduckgo")], #google backend frequently gave connection errors, so different backends were tested until an operational one was found
    instructions=[
        "When asked, search the internet for up to date information on the topic",
        "If a search returns nothing relevant, say so — do not invent stories.",
    ],
    markdown=True,
    debug_mode=True,
)


DEBUG Function: web_search registered with websearch

DEBUG Function: search_news registered with websearch

In [12]:
scout.print_response("What date is it. In order to fulfill this request, query google with the question, and then return the first response to me", stream=True)

DEBUG Session ID: 43f3f342-4d1b-49e6-a836-d6095c646ea8

DEBUG Agent initialized: scout

DEBUG Creating new AgentSession: 43f3f342-4d1b-49e6-a836-d6095c646ea8

DEBUG Agent Run Start: 28473943-a085-4305-895f-13df6b1035e2

DEBUG Processing tools for model

DEBUG Added tool web_search from websearch

... (27 outputs left out)

DEBUG  METRICS

DEBUG Ollama Response Stream End

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: 28473943-a085-4305-895f-13df6b1035e2

## Questions on Web Search Agent

**What to do:** in the code cell below, ask `scout` the **same up-to-date questions** you asked `plain_agent`
in Part 1 (copy the `scout.print_response(...)` line from above and change the question).
Because `debug_mode=True`, you can see in the output whether the agent actually called the web search tool — look for it.

> **How to answer:** questions marked ✍️ are **written** — double-click the *Answers* cell below and type your answer after the colon.
> Questions marked 💻 need **code** — copy the relevant cell from above into the empty code cell provided, edit it, and run it.
> Save your notebook (**File → Save** / Ctrl+S) when you are done with each part.

1. ✍️ How does the behaviour compare with `plain_agent` for the same query?
2. ✍️ Does the agent ever find the correct information? Did it call the search tool every time?
3. ✍️ Does the agent hallucinate as much?
4. ✍️ If the agent still can't reliably retrieve up-to-date information, even with internet access, how could we improve this? (hint: visit https://ollama.com/library and look at which models support **tools**)

In [13]:
# 💻 Ask scout the same up-to-date questions you asked plain_agent
scout.print_response("What was the main contributing reason to the fall of Napoleon III", stream=True)

DEBUG Session ID: 43f3f342-4d1b-49e6-a836-d6095c646ea8

DEBUG Agent initialized: scout

DEBUG Creating new AgentSession: 43f3f342-4d1b-49e6-a836-d6095c646ea8

DEBUG Agent Run Start: ae926471-cbb0-4b7c-9d40-85bf62ccb3ce

DEBUG Processing tools for model

DEBUG Added tool web_search from websearch

... (27 outputs left out)

DEBUG  METRICS

DEBUG Ollama Response Stream End

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: ae926471-cbb0-4b7c-9d40-85bf62ccb3ce

In [14]:
scout.print_response("When was the most recent Irish soccer match, and what was the result", stream=True)

DEBUG Session ID: 43f3f342-4d1b-49e6-a836-d6095c646ea8

DEBUG Agent initialized: scout

DEBUG Creating new AgentSession: 43f3f342-4d1b-49e6-a836-d6095c646ea8

DEBUG Agent Run Start: e8a6b49e-c8a6-406b-9547-4703d529618b

DEBUG Processing tools for model

DEBUG Added tool web_search from websearch

... (27 outputs left out)

DEBUG  METRICS

DEBUG Ollama Response Stream End

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: e8a6b49e-c8a6-406b-9547-4703d529618b

In [15]:
scout.print_response("What valuation was placed on Ironwood Labs when it IPO'd in 2021?", stream=True)

DEBUG Session ID: 43f3f342-4d1b-49e6-a836-d6095c646ea8

DEBUG Agent initialized: scout

DEBUG Creating new AgentSession: 43f3f342-4d1b-49e6-a836-d6095c646ea8

DEBUG Agent Run Start: 818da06c-f323-41f7-b34a-4cd03a8ef3e2

DEBUG Processing tools for model

DEBUG Added tool web_search from websearch

... (27 outputs left out)

DEBUG  METRICS

DEBUG Ollama Response Stream End

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: 818da06c-f323-41f7-b34a-4cd03a8ef3e2

### Answers — Web Search Agent
1. Behaviour compared with plain_agent:
* Gave shorter answers to the general knowledge question, but usually answered the question better
* Was able to look up recent soccer matches, where the plain agent was not, although the data it returned was a few days out of date.
* Found a company that happened to match the name of the fictitiuos company, and gave information relevant to that company
3. Did it find the correct information / did it call the tool:
* tool was called every time, even when not quite necessary, and it mostly found correct information
4. Does it hallucinate as much: No
5. How could we improve this: Give the agent greater ability to scrape deaply into web info, as it would have have found the latest game info if it could, rather than the second latest

### Custom Tools Continued

As mentioned, toolls can either be part of a toolkit, or they can be compeltely custom functionality. 
In this example we are going create a custom tool which simply returns the number of employees in our fictional company "Ironwood Labs" roster. 


First, we need to define our roster. 
Next, we create our deterministic python function to parse the text for employee entry count. 
Finally, we feed this tool to the Agent to use when it needs.

### The roster

In [16]:
from pathlib import Path

corpus_dir = Path("/kaggle/working/corpus")
corpus_dir.mkdir(parents=True, exist_ok=True)
# Updated about.md: "42 people", three research divisions plus a small Operations team
(corpus_dir / "about.md").write_text('''# About Ironwood Labs

Ironwood Labs is a materials-science research company founded in 2018 in
Reykjavík, Iceland. Our mission is to develop novel composites that survive
the temperature and radiation cycles of low-earth orbit and the deep ocean.

We are a team of 42 people, organised into three research divisions plus a
small Operations team:

- **Composites Division** — led by Dr. Amara Osei
- **Thermal Systems Division** — led by Dr. Kenji Watanabe
- **Field Deployment Division** — led by Dr. Ines Marchetti
- **Operations** — facilities, procurement and administration

The full staff list is maintained in roster.md.

Ironwood is privately funded and does not accept government contracts.
Our current annual research budget is €18M.
''')

(corpus_dir / "roster.md").write_text('''# Ironwood Labs Staff Roster

| Name | Division | Role |
|------|----------|------|
| Dr. Amara Osei | Composites | Division Lead |
| Liam Fitzgerald | Composites | Senior Materials Scientist |
| Sofia Almeida | Composites | Materials Scientist |
| Hiroshi Tanaka | Composites | Materials Scientist |
| Grace Mwangi | Composites | Lattice Fabrication Engineer |
| Oskar Johansson | Composites | Lattice Fabrication Engineer |
| Priya Raman | Composites | Mechanical Test Engineer |
| Daniel Okafor | Composites | Mechanical Test Engineer |
| Elin Sigurdardottir | Composites | Research Associate |
| Mateo Rossi | Composites | Research Associate |
| Chloe Bernard | Composites | Microscopy Specialist |
| Aditya Kulkarni | Composites | Postdoctoral Researcher |
| Freya Halvorsen | Composites | Postdoctoral Researcher |
| Tomás Herrera | Composites | Lab Technician |
| Dr. Kenji Watanabe | Thermal Systems | Division Lead |
| Anika Schulz | Thermal Systems | Senior Thermal Engineer |
| Rafael Mendes | Thermal Systems | Thermal Engineer |
| Leila Haddad | Thermal Systems | Radiation Physicist |
| Björn Einarsson | Thermal Systems | Radiation Physicist |
| Nadia Petrova | Thermal Systems | Coatings Engineer |
| Samuel Adeyemi | Thermal Systems | Coatings Engineer |
| Julia Kowalska | Thermal Systems | Simulation Engineer |
| Arjun Mehta | Thermal Systems | Simulation Engineer |
| Ingrid Solberg | Thermal Systems | Postdoctoral Researcher |
| Marco Bianchi | Thermal Systems | Research Associate |
| Hana Kobayashi | Thermal Systems | Research Associate |
| Eamon Gallagher | Thermal Systems | Lab Technician |
| Dr. Ines Marchetti | Field Deployment | Division Lead |
| Petra Nowak | Field Deployment | Deployment Scheduler |
| Yusuf Bello | Field Deployment | Equipment Logistics Coordinator |
| Kai Andersen | Field Deployment | Marine Systems Engineer |
| Lucia Moreno | Field Deployment | Marine Systems Engineer |
| Femi Adebayo | Field Deployment | ROV Operator |
| Astrid Lund | Field Deployment | ROV Operator |
| Noah Brennan | Field Deployment | Oceanographer |
| Zara Hussain | Field Deployment | Biofouling Analyst |
| Gunnar Olafsson | Field Deployment | Field Technician |
| Mei Lin | Field Deployment | Data Analyst |
| Marta Lindqvist | Operations | Facilities Manager |
| Sigrid Magnusdottir | Operations | Procurement Officer |
| Omar Farouk | Operations | IT Administrator |
| Rosa Delgado | Operations | Office Administrator |
''')

2457

### The tool

In [17]:
from pathlib import Path

ROSTER_PATH = Path("/kaggle/working/corpus/roster.md")

def count_employees() -> int:
    """Return the total number of employees at Ironwood Labs, counted from the staff roster."""
    table_rows = [l for l in ROSTER_PATH.read_text().splitlines() if l.startswith("|")]
    return len(table_rows) - 2  # drop the header row and the |---| separator row

In [18]:
hr = Agent(
    
    name="HR",
    role="Answers questions about Ironwood Labs staff using internal HR tools",
    
    model=Ollama(id="llama3.2"),
    
    tools=[count_employees],
    instructions=[
        "You answer questions about Ironwood Labs employees.",
        "For any question about how many people work at Ironwood, call the count_employees tool.",
        "Never guess or estimate staff numbers; only report what the tool returns.",
        "If no tool can answer the question, say so instead of making something up.",
    ],
    markdown=True,
    debug_mode=True,
)


In [19]:
hr.print_response("How many employees does Ironwood Labs have?", stream=True)

DEBUG Session ID: 208cbeb8-7199-47fa-8a3d-afad78446932

DEBUG Agent initialized: hr

DEBUG Creating new AgentSession: 208cbeb8-7199-47fa-8a3d-afad78446932

DEBUG Agent Run Start: c2158566-7abd-41e1-bf74-cec8b474bd32

DEBUG Processing tools for model

DEBUG Added tool count_employees

... (25 outputs left out)

DEBUG  METRICS

DEBUG Ollama Response Stream End

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: c2158566-7abd-41e1-bf74-cec8b474bd32

### Questions on Custom Tools

We now have a custom tool that parses the employee roster at runtime and returns a count — about as simple as a tool can be!
(You can write new documents / extra roster entries if it helps you answer.)

> **How to answer:** questions marked ✍️ are **written** — double-click the *Answers* cell below and type your answer after the colon.
> Questions marked 💻 need **code** — copy the relevant cell from above into the empty code cell provided, edit it, and run it.
> Save your notebook (**File → Save** / Ctrl+S) when you are done with each part.

1. ✍️ Why would we want such a simple custom tool? Can an LLM not do this itself? (hint: what if we had thousands of employees?)
2. ✍️ What other tools could we design here to offload work from the LLM to deterministic Python functions? List at least two, with a one-line description of each.
3. 💻 Implement one of your tools from Q2 in the code cell below — copy the `count_employees` and `hr` cells from above, write your new function, add it to the agent's `tools=[...]` list, and ask a question that uses it.

In [20]:
# 💻 Q3: your own custom tool
# 1. Write your function (give it a clear docstring - the agent reads it!)
# 2. Copy the hr agent cell from above and add your function to tools=[...]
# 3. Ask a question that should trigger it

from pathlib import Path

ROSTER_PATH = Path("/kaggle/working/corpus/roster.md")

def employees_per_division(division: str) -> int:
    """Return the number of Ironwood Labs employees in the given division."""
    target = division.strip().casefold()
    count = 0

    for line in ROSTER_PATH.read_text().splitlines():
        if not line.strip().startswith("|"):
            continue

        cells = [cell.strip() for cell in line.strip().strip("|").split("|")]

        # Skip the header, separator, and malformed rows.
        if len(cells) != 3 or cells[0] == "Name":
            continue
        if all(cell and set(cell) <= {"-", ":"} for cell in cells):
            continue

        if cells[1].casefold() == target:
            count += 1

    return count

In [21]:
hr = Agent(
    
    name="HR",
    role="Answers questions about Ironwood Labs staff using internal HR tools",
    
    model=Ollama(id="llama3.2"),
    
    tools=[count_employees, employees_per_division],
    instructions=[
        "You answer questions about Ironwood Labs employees.",
        "For any question about how many people work at Ironwood, call the count_employees tool.",
        "Never guess or estimate staff numbers; only report what the tool returns.",
        "If no tool can answer the question, say so instead of making something up.",
    ],
    markdown=True,
    debug_mode=True,
)

In [22]:
hr.print_response('How many employees in the composites division?', stream=True)

DEBUG Session ID: e7544f67-e196-4093-ad6b-306702963599

DEBUG Agent initialized: hr

DEBUG Creating new AgentSession: e7544f67-e196-4093-ad6b-306702963599

DEBUG Agent Run Start: 7f8e8eef-0bbe-43ff-938f-dc6e79034a9d

DEBUG Processing tools for model

DEBUG Added tool count_employees

... (26 outputs left out)

DEBUG  METRICS

DEBUG Ollama Response Stream End

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: 7f8e8eef-0bbe-43ff-938f-dc6e79034a9d

### Answers — Custom Tools
1. Why use such a simple tool / can the LLM not do it itself: The LLM could hallucinate the answer, while the custom-made tool that specifically counts the number of employees should reduce the chance of hallucinations.
2. Other tools we could design (at least two):
* A tool which counts the number of employees in each division.
* A tool which returns the roll and division of an employee when passed the name as a parameter.
4. See the code cell above.

## Part 3 — Retrival Augemented Generation

While websearch is great for retriving new and relevant *general* information. What if we have information that only we know? Private internal company info for example, or even lecture notes. We cant expect this information to already be on the internet - and as an extension in the models learned "memory".

So what is our potential local solution?


With Retrival Augemented Generation (RAG), rather than search the internet for info, we inject our context window with relevant information from our own Document store - thus Augmenting the Generation. RAG in its simplest form can be broken up into two key models that interact (think of an agent and its tools again, this time its almost like we supply another model as a tool).

RAG can be broken into two steps:
* Retrival: An embedding model processes the input Query and produces an embedding vector (MiniLM-L6-v2 is a tiny well known embedding model: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2). The cosine similarity between this vector and the precomputed vectors of all our document chunks denotes how semantically similar the document chunks are to the query (we use the same embedding model on all our docs offline and store these vectors for runtime comparison)
* Augmented Generation: Once the N document chunks have been selected by the Retrival model, these are loaded into the Context Window of our agent / LLM - the Augemented Generator. 


In this way, we can inject relevant information from our documents etc. into the model at run time.



Practical worked example - back to Ironwood labs


We need the Librarian agent to know about Ironwood's internal documents. In real life these might be Confluence pages, PDFs, Notion docs. For this lab, we'll create a small handful of Markdown files in a `corpus/` directory. Importantly, our Ironwood labs is entirely fictional.

**Why fictional?** Because when you ask a question, you'll be able to tell *immediately* whether the agent is grounding its answer in the docs or making something up - the info we are looking for only exists in this notebook.

### Step 1 - The saved Document store:
We are going to:

1. Chunk each document into pieces small enough to fit in the model's context.
2. Embed each chunk — turn it into a vector (a list of ~384 numbers) using a small local model. Chunks about similar topics end up near each other in vector space.
3. Store those vectors in a vector database (LanceDB, running locally).
4. When the user asks a question, we embed the question the same way, find the closest chunks in the database, and paste them into the model's prompt. (This behaviour is handled by the Agno agent harness as we will see later)

### The Documents (Corpus)

In [23]:
from pathlib import Path

corpus_dir = Path("/kaggle/working/corpus")
corpus_dir.mkdir(parents=True, exist_ok=True)

(corpus_dir / "about.md").write_text('''# About Ironwood Labs

Ironwood Labs is a materials-science research company founded in 2018 in
Reykjavík, Iceland. Our mission is to develop novel composites that survive
the temperature and radiation cycles of low-earth orbit and the deep ocean.

We are a team of 42 people, organised into three research divisions plus a
small Operations team:

- **Composites Division** — led by Dr. Amara Osei
- **Thermal Systems Division** — led by Dr. Kenji Watanabe
- **Field Deployment Division** — led by Dr. Ines Marchetti
- **Operations** — facilities, procurement, IT and administration

Our headquarters and main laboratories are in Reykjavík. We also operate a
marine test station in Ísafjörður, in the Westfjords, which houses our
hydrostatic pressure testing equipment.

The full staff list is maintained in roster.md.

Ironwood is privately funded and does not accept government contracts.
Our current annual research budget is €18M.
''')

(corpus_dir / "history.md").write_text('''# Ironwood Labs Company History

**2018** — Ironwood Labs is founded in Reykjavík by Dr. Amara Osei and
Dr. Henrik Dahl, with five staff and seed funding from Nordvik Capital.
Dr. Osei leads the Composites Division and Dr. Dahl leads Thermal Systems.

**2019** — Kestrel-1, our first commercial lattice material, is released.

**2020** — Dr. Ines Marchetti joins from a deep-sea robotics background and
founds the Field Deployment Division to take materials out of the lab and
into real environments.

**2021** — Kestrel-2 is released. Dr. Kenji Watanabe joins Thermal Systems
as Principal Engineer.

**2022** — The Marlin-M marine composite launches. The Ísafjörður marine
test station opens. A dedicated Operations team is formed to take over
facilities, procurement and administration from the research divisions.

**2023** — Dr. Henrik Dahl retires. Dr. Kenji Watanabe succeeds him as lead
of the Thermal Systems Division. Dr. Dahl remains an external scientific
advisor but is no longer an employee.

**2024** — Kestrel-3 is released. The Composites Division publishes its
work on recursive lattice healing.

**2025** — Cormorant-C7 enters qualification testing for a lunar surface
application with Selene Dynamics.
''')

(corpus_dir / "products.md").write_text('''# Ironwood Product Lines

## Kestrel Series
The Kestrel is our flagship carbon-titanium lattice material, developed by
the Composites Division. Product lead: Liam Fitzgerald.

- **Kestrel-1** (2019): tensile strength 860 MPa, stable between -90°C and
  +280°C. Discontinued in 2023.
- **Kestrel-2** (2021): tensile strength 980 MPa, stable between -120°C and
  +350°C. Still supplied for legacy customers.
- **Kestrel-3** (2024): tensile strength 1,240 MPa, stable between -180°C
  and +420°C. Used in three orbital satellite platforms built by Halden
  Orbital.

## Marlin Series
Marlin composites are designed for high-pressure marine environments. The
material is developed by Composites and field-tested by the Field
Deployment Division. Product lead: Kai Andersen.

- **Marlin-M**: rated to 6,000m depth. Deployed in two long-duration
  seafloor observatories in the Puysegur Trench, operated by the Southern
  Ocean Observatory Consortium.
- **Marlin-S**: a lower-cost shallow-water variant rated to 1,500m, used
  for offshore sensor housings.

## Cormorant Coatings
Cormorant is our thin-film radiation shielding, applied via vacuum
deposition and developed by the Thermal Systems Division.
Product lead: Nadia Petrova.

- **Cormorant-C5**: equivalent shielding to 2.5mm of aluminum at 18% of the
  mass.
- **Cormorant-C7**: equivalent shielding to 4mm of aluminum at 12% of the
  mass. Currently in qualification testing for a lunar surface application
  with Selene Dynamics.
''')

(corpus_dir / "research.md").write_text('''# Recent Research at Ironwood (2024-2025)

## Composites Division
In Q4 2024, the Composites Division published findings on "recursive lattice
healing" — a mechanism by which Kestrel-3 recovers approximately 78% of its
original tensile strength after micro-fracture events, without external
intervention. This is attributed to unreacted titanium precursors migrating
to fracture surfaces under mechanical stress.

Current project: a Kestrel-4 prototype targeting 1,500 MPa tensile strength,
led by Liam Fitzgerald. First samples are expected in mid-2026.

## Thermal Systems Division
The Thermal Systems Division is currently investigating why Cormorant-C7
samples show a 6% shielding improvement when exposed to sustained cosmic
ray flux for over 90 days. Early hypotheses point to defect annealing
under low-dose radiation. This work is led by radiation physicist
Leila Haddad.

## Field Deployment Division
Field Deployment reported in March 2025 that Marlin-M panels recovered
from the Puysegur Trench after 14 months showed less biofouling than
expected — approximately 40% less than steel controls at the same depth.
Biofouling analyst Zara Hussain is now running a follow-up study comparing
Marlin-M and Marlin-S panels at the Ísafjörður station.
''')

(corpus_dir / "partners.md").write_text('''# Ironwood Partners and Customers

**Halden Orbital** (Tromsø, Norway) — Satellite manufacturer and our largest
customer. Kestrel-3 is used as structural lattice in three of their orbital
platforms. Technical contact: Liam Fitzgerald.

**Southern Ocean Observatory Consortium (SOOC)** — Operates the two seafloor
observatories in the Puysegur Trench where Marlin-M is deployed. Deployment
scheduling is coordinated by Petra Nowak.

**Selene Dynamics** — Lunar lander company. Cormorant-C7 is in qualification
testing for use on their lunar surface habitat panels. A pass/fail decision
is expected by Q2 2026. Technical contact: Nadia Petrova.

**University of Iceland** — Joint PhD programme in materials science. We
fund up to two PhD students per year, co-supervised by an Ironwood
division lead.
''')

(corpus_dir / "facilities.md").write_text('''# Ironwood Facilities and Equipment

All equipment bookings go through Marta Lindqvist (Operations) unless noted.

## Reykjavík HQ — Building B Laboratories

**Vacuum deposition chamber** (room B2) — Used to apply Cormorant coatings.
Booked in 4-hour blocks; requests must be submitted 72 hours in advance.
Only trained operators may run it: Nadia Petrova and Samuel Adeyemi.

**Thermal cycling chamber** (room B4) — Cycles samples between -196°C and
+500°C. Maintained by Eamon Gallagher.

**Radiation test cell** (room B6) — Gamma and proton exposure testing.
Leila Haddad is the Radiation Safety Officer. Separate radiation safety
training is required before first use, and a personal dosimeter must be
worn at all times.

**Scanning electron microscope** (room B3) — Managed by Chloe Bernard.
Book directly with her rather than through Operations.

**Mechanical test lab** (room B1) — Tensile and fatigue testing rigs.
Managed by Priya Raman.

## Ísafjörður Marine Test Station

**Hydrostatic pressure vessel** — Simulates depths up to 7,000m. Used for
all Marlin qualification testing. The station is run by field technician
Gunnar Olafsson.
''')

(corpus_dir / "policies.md").write_text('''# Ironwood Policies

## Publications
All external publications and conference talks must be approved by the
author's division lead and go through a 30-day pre-publication review to
check for patentable material.

## Remote Work
Staff may work remotely up to two days per week. Lab-based roles (lab
technicians, field technicians and equipment operators) are exempt and
work on site.

## Travel
All business travel is booked through Rosa Delgado (Operations). Field
deployment travel must also be approved by Dr. Ines Marchetti.

## Procurement
Purchases over €5,000 require approval from both Sigrid Magnusdottir
(Procurement) and the requesting division lead.

## Lab Safety
All new staff must complete the general lab safety induction in their
first week. Radiation test cell access requires additional training.

## Visiting Researchers
We accept up to 6 visiting researchers per year on 3–6 month engagements.
Applications open each January and close at the end of February. Every
visitor must have a named host from Ironwood's senior staff.

## Government Work
Ironwood is privately funded and does not accept government contracts or
government grants of any kind.
''')

(corpus_dir / "team_faq.md").write_text('''# Ironwood Team FAQ

**Who should I contact about Kestrel material samples?**
Dr. Amara Osei (Composites Division). Sample requests must include the
intended application and destructive testing plan.

**How do I book time on the vacuum deposition chamber?**
Contact Marta Lindqvist in Operations. The chamber is booked in 4-hour
blocks; requests must be submitted 72 hours in advance.

**Who handles field deployment logistics for oceanographic partners?**
Dr. Ines Marchetti's team, specifically Petra Nowak for scheduling and
Yusuf Bello for equipment shipping.

**What is Ironwood's policy on academic collaboration?**
We accept up to 6 visiting researchers per year on 3–6 month engagements.
Applications open each January.

**My computer or network access isn't working. Who do I ask?**
Omar Farouk, our IT Administrator.

**How do I order lab supplies?**
Submit a request to Sigrid Magnusdottir. Orders over €5,000 also need
your division lead's approval.

**Who is the Radiation Safety Officer?**
Leila Haddad (Thermal Systems).
''')

(corpus_dir / "roster.md").write_text('''# Ironwood Labs Staff Roster

| Name | Division | Role |
|------|----------|------|
| Dr. Amara Osei | Composites | Division Lead |
| Liam Fitzgerald | Composites | Senior Materials Scientist |
| Sofia Almeida | Composites | Materials Scientist |
| Hiroshi Tanaka | Composites | Materials Scientist |
| Grace Mwangi | Composites | Lattice Fabrication Engineer |
| Oskar Johansson | Composites | Lattice Fabrication Engineer |
| Priya Raman | Composites | Mechanical Test Engineer |
| Daniel Okafor | Composites | Mechanical Test Engineer |
| Elin Sigurdardottir | Composites | Research Associate |
| Mateo Rossi | Composites | Research Associate |
| Chloe Bernard | Composites | Microscopy Specialist |
| Aditya Kulkarni | Composites | Postdoctoral Researcher |
| Freya Halvorsen | Composites | Postdoctoral Researcher |
| Tomás Herrera | Composites | Lab Technician |
| Dr. Kenji Watanabe | Thermal Systems | Division Lead |
| Anika Schulz | Thermal Systems | Senior Thermal Engineer |
| Rafael Mendes | Thermal Systems | Thermal Engineer |
| Leila Haddad | Thermal Systems | Radiation Physicist |
| Björn Einarsson | Thermal Systems | Radiation Physicist |
| Nadia Petrova | Thermal Systems | Coatings Engineer |
| Samuel Adeyemi | Thermal Systems | Coatings Engineer |
| Julia Kowalska | Thermal Systems | Simulation Engineer |
| Arjun Mehta | Thermal Systems | Simulation Engineer |
| Ingrid Solberg | Thermal Systems | Postdoctoral Researcher |
| Marco Bianchi | Thermal Systems | Research Associate |
| Hana Kobayashi | Thermal Systems | Research Associate |
| Eamon Gallagher | Thermal Systems | Lab Technician |
| Dr. Ines Marchetti | Field Deployment | Division Lead |
| Petra Nowak | Field Deployment | Deployment Scheduler |
| Yusuf Bello | Field Deployment | Equipment Logistics Coordinator |
| Kai Andersen | Field Deployment | Marine Systems Engineer |
| Lucia Moreno | Field Deployment | Marine Systems Engineer |
| Femi Adebayo | Field Deployment | ROV Operator |
| Astrid Lund | Field Deployment | ROV Operator |
| Noah Brennan | Field Deployment | Oceanographer |
| Zara Hussain | Field Deployment | Biofouling Analyst |
| Gunnar Olafsson | Field Deployment | Field Technician |
| Mei Lin | Field Deployment | Data Analyst |
| Marta Lindqvist | Operations | Facilities Manager |
| Sigrid Magnusdottir | Operations | Procurement Officer |
| Omar Farouk | Operations | IT Administrator |
| Rosa Delgado | Operations | Office Administrator |
''')

print("Corpus files created:")
for f in sorted(corpus_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size} bytes)")


Corpus files created:
  about.md  (950 bytes)
  facilities.md  (1167 bytes)
  history.md  (1241 bytes)
  partners.md  (808 bytes)
  policies.md  (1174 bytes)
  products.md  (1492 bytes)
  research.md  (1261 bytes)
  roster.md  (2459 bytes)
  team_faq.md  (1040 bytes)


###  Embedding the Document store


In [24]:
from agno.knowledge.knowledge import Knowledge
from agno.knowledge.embedder.sentence_transformer import SentenceTransformerEmbedder
from agno.vectordb.lancedb import LanceDb

# 384-dim embeddings, ~23 MB, runs on CPU. Perfect for a lab.
embedder = SentenceTransformerEmbedder(id="sentence-transformers/all-MiniLM-L6-v2")

ironwood_knowledge = Knowledge(
    vector_db=LanceDb(
        uri="/kaggle/working/lancedb",
        table_name="ironwood_docs",
        embedder=embedder,
    ),
)

# Insert each document. Agno chunks + embeds + stores in one call.
for doc in sorted(corpus_dir.glob("*.md")):
    print(f"Inserting {doc.name}...")
    ironwood_knowledge.insert(path=str(doc))

print("\n✅ Knowledge base ready.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (79 outputs left out)

DEBUG Inserting 1 documents

DEBUG Parsed document: team_faq.md ({'chunk': 1, 'chunk_size': 1029, 'linked_to': ''})

DEBUG Inserted 1 documents


✅ Knowledge base ready.


### Step 2 - the context Grounded Agent:
Now we have the document store ready to go, we attach this "knowledge"  to the Agno agent. Other harnesses / packages will have their own method and naming for this, also if you really wanted to this could be one of the easier agentic concepts to implement by hand (programming the series of retrival, similarity then agent, rather than letting the harness handle it automatically).

Now we should be able to query our Librarian agent for infomation that only exists internally. 

This technique can still be useful for retriving information which is available already on the internet as well too - adding the grounded accurate information to context helps improve performance on many tasks!

In [25]:
from agno.agent import Agent
from agno.models.ollama import Ollama

librarian = Agent(
    name="Librarian",
    role="Answer questions using Ironwood Labs' internal documentation",
    
    model=Ollama(id="llama3.2"),

    knowledge=ironwood_knowledge,
    instructions=[
        "Answer using the retrieved Ironwood documents.",
        "Quote the specific document when citing a fact (e.g. 'per team_faq.md').",
        "If the documents do not contain the answer, say so plainly. Do NOT guess.",
    ],
    search_knowledge=True,   # let the agent decide when to search
    markdown=True,
    debug_mode=True,

)

In [26]:
librarian.print_response("Who leads the Composites Division?", stream=True)

DEBUG Session ID: 3dbd11fd-e34c-4e60-b870-df2c37625512

DEBUG Agent initialized: librarian

DEBUG Creating new AgentSession: 3dbd11fd-e34c-4e60-b870-df2c37625512

DEBUG Agent Run Start: 86f7230d-4061-40c2-9943-7c5ecfbd5a87

DEBUG Processing tools for model

DEBUG Added tool search_knowledge_base

... (29 outputs left out)

DEBUG  METRICS

DEBUG Ollama Response Stream End

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: 86f7230d-4061-40c2-9943-7c5ecfbd5a87

## Questions on RAG Agent

**What to do:** in the first code cell below, ask `librarian` **at least 4 questions** — copy the
`librarian.print_response(...)` line from above and change the question. Include:
- questions whose answers are only in our Ironwood documents (e.g. *"Who is the Radiation Safety Officer?"*)
- at least one question that is **not** in the documents (e.g. *"What is Ironwood's share price?"*)
- at least one question that needs facts from **two** documents (e.g. *"Who leads the division that makes Cormorant coatings?"*)

Check the answers against the documents in the corpus cell above.

> **How to answer:** questions marked ✍️ are **written** — double-click the *Answers* cell below and type your answer after the colon.
> Questions marked 💻 need **code** — copy the relevant cell from above into the empty code cell provided, edit it, and run it.
> Save your notebook (**File → Save** / Ctrl+S) when you are done with each part.

1. ✍️ Where does the model get the information it uses to answer your query? (hint: look at the debug output)
2. ✍️ How does the model handle queries that are outside its knowledge base?
3. ✍️ If you wanted to add more information — or edit existing information — how would you do this?
4. ✍️ Does the agent always answer questions about the documents correctly? Why / why not? Is this a concern?
5. 💻 In the second code cell below, copy the `librarian` agent cell and add our `count_employees` tool (`tools=[count_employees]`) plus an instruction telling it when to use the tool. Ask *"How many employees does Ironwood Labs have?"*
   - ✍️ Look at the system prompt in the debug output before and after adding the tool — what changed?
   - ✍️ When the query only asks about employee count, does the agent still search the documents (RAG), use the tool, or both?

In [27]:
# 💻 Q1-4: ask librarian your questions here
librarian.print_response("Who is the majority owner of Ironwood Labs", stream=False)

librarian.print_response("What department is in charge of the hydrostatic pressure vessel", stream=False)

librarian.print_response("When was Ironwood Labs founded", stream=False)

librarian.print_response("When did the icelandic government announce its plan to shut down Ironwood Labs for violating health and safety regulations", stream=False)


DEBUG Session ID: 3dbd11fd-e34c-4e60-b870-df2c37625512

DEBUG Agent initialized: librarian

DEBUG Creating new AgentSession: 3dbd11fd-e34c-4e60-b870-df2c37625512

DEBUG Agent Run Start: e01fc40c-e030-42d7-883d-0cbbe89fd2f2

DEBUG Processing tools for model

DEBUG Added tool search_knowledge_base

... (141 outputs left out)

DEBUG Tool call Id: 5ec10569-b420-4876-bcc1-b58602c3a043

DEBUG [                                                                                                            
        {                                                                                                          
          "content": "# Ironwood Policies\n## Publications\nAll external publications and conference talks must be 
      approved by the\nauthor's division lead and go through a 30-day pre-publication review to\ncheck for         
      patentable material.\n## Remote Work\nStaff may work remotely up to two days per week. Lab-based roles       
      (lab\ntechnicians, field technicians and equipment operators) are exempt and\nwork on site.\n## Travel\nAll  
      business travel is booked through Rosa Delgado (Operations). Field\ndeployment travel must also be approved  
      by Dr. Ines Marchetti.\n## Procurement\nPurchases over €5,000 require approval from both Sigrid              
      Magnusdottir\n(Procurement) and the requesting division lead.\n## Lab Safety\nAll new staff must complete the
      general lab safety induction in their\nfirst week. Radiation test cell access requires additional            
      training.\n## Visiting Researchers\nWe accept up to 6 visiting researchers per year on 3–6 month             
      engagements.\nApplications open each January and close at the end of February. Every\nvisitor must have a    
      named host from Ironwood's senior staff.\n## Government Work\nIronwood is privately funded and does not      
      accept government contracts or\ngovernment grants of any kind.\n",                                           
          "name": "policies.md",                                                                                   
          "meta_data": {                                                                                           
            "chunk": 1,                                                                                            
            "chunk_size": 1163,                                                                                    
            "linked_to": ""                                                                                        
          }                                                                                                        
        },                                                                                                         
        {                                                                                                          
          "content": "# About Ironwood Labs\nIronwood Labs is a materials-science research company founded in 2018 
      in\nReykjavík, Iceland. Our mission is to develop novel composites that survive\nthe temperature and         
      radiation cycles of low-earth orbit and the deep ocean.\nWe are a team of 42 people, organised into three    
      research divisions plus a\nsmall Operations team:\n- **Composites Division** — led by Dr. Amara Osei\n-      
      **Thermal Systems Division** — led by Dr. Kenji Watanabe\n- **Field Deployment Division** — led by Dr. Ines  
      Marchetti\n- **Operations** — facilities, procurement, IT and administration\nOur headquarters and main      
      laboratories are in Reykjavík. We also operate a\nmarine test station in Ísafjörður, in the Westfjords, which
      houses our\nhydrostatic pressure testing equipment.\nThe full staff list is maintained in                    
      roster.md.\nIronwood is privately funded and does not accept government contracts.\nOur current annual       
      research budget is €18M.\n",                                                                                 
          "name": "about.md",                                                                                      
          "meta_data": {                                                                                           
            "chunk": 1,                                 

DEBUG Run a7114371-a969-4b07-8721-2ce92f199545 was cancelled

DEBUG Added RunOutput to Agent Session

In [40]:
# 💻 Q5: copy the librarian agent cell here, add tools=[count_employees] and an instruction about when to use it,
# then ask: "How many employees does Ironwood Labs have?"
from agno.agent import Agent
from agno.models.ollama import Ollama

librarian = Agent(
    name="Librarian",
    role="Answer questions using Ironwood Labs' internal documentation",
    
    model=Ollama(id="llama3.2"),
    tools=[count_employees],

    knowledge=ironwood_knowledge,
    instructions=[
        "Answer using the retrieved Ironwood documents.",
        "Quote the specific document when citing a fact (e.g. 'per team_faq.md').",
        "If the documents do not contain the answer, say so plainly. Do NOT guess.",
    ],

    
    search_knowledge=True,   # let the agent decide when to search
    markdown=True,
    debug_mode=True,

)

In [38]:
librarian.print_response('how many employees does ironwood labs have?', stream=True)

DEBUG Session ID: 98756cad-6701-437e-8d3a-af5f9635f312

DEBUG Agent initialized: librarian

DEBUG Creating new AgentSession: 98756cad-6701-437e-8d3a-af5f9635f312

DEBUG Agent Run Start: bd14ddfa-f629-4253-9429-4a66058acf86

DEBUG Processing tools for model

DEBUG Added tool count_employees

... (5 outputs left out)

DEBUG =========================== user ===========================

DEBUG how many employees does ironwood labs have?

ERROR    Error in Agent run: [Errno 111] Connection refused

DEBUG Added RunOutput to Agent Session

### Answers — RAG Agent
1. Where the model gets its information from: the ironwood_knowledge database, with all the company documents encoded within it
2. How it handles queries outside its knowledge base: it says that it doesn't have the information to answer within the knowledge base, and gives a half-answer
3. How to add or edit information: make additional knowledge docs or edit existing knowledge documents and encode then into the ironwood_knowledge database
4. Does it always answer correctly / why / is this a concern: No, it failed to correctly answer the number of employees. This is an issue, as it shows that the models will not always correctly return information to the user.
5. See the code cell above.
   - What changed in the system prompt after adding the tool: It had to read through the entire database, as opposed to just the roster.md
   - For an employee-count-only query, did it use RAG, the tool, or both: It ran the tool, and skipped the using RAG, completing the task faster as a result

## Final Question

Can we combine tools, internet search and RAG into one capable agent?

**What to do:** using what you have learned above, build one agent in the code cell below — start by copying the
`librarian` cell and add to it. Your agent should:

1. 💻 Use our RAG document store when necessary (`knowledge=ironwood_knowledge`).
2. 💻 Rely on tool calls where appropriate, e.g. counting employees (`count_employees`).
3. 💻 Still have internet capability (`WebSearchTools()`).

Test it with **one question of each kind** (document question, employee count, up-to-date web question) and check which source it used in the debug output.

4. 💻 Swap in a stronger model and compare: pull a bigger tool-capable model from https://ollama.com/library
   (e.g. `!ollama pull qwen3:8b`), change `Ollama(id=...)`, and re-ask the same three questions.
5. 💻 + ✍️ Enable the GPU (Kaggle: **Settings → Accelerator → GPU**, then re-run Part 0 setup) and time the same query on CPU vs GPU.
   Why is it faster? What limits the size of model we can run?

## Answers

5. It is faster because using the GPU enables parrallelisation of tasks, resulting in the overall process executing faster compared to the CPU. THe size of the model that can be loaded is limited by the amount of VRAM available on the GPUs.

In [30]:
# 💻 Q1-3: build your combined agent here (copy the librarian cell and add tools + web search)
from agno.tools.websearch import WebSearchTools

omnipotens = Agent(
    name="Librarian",
    role="Answer questions using Ironwood Labs' internal documentation",
    
    model=Ollama(id="llama3.2"),
    tools=[count_employees, employees_per_division, WebSearchTools(backend="duckduckgo")],

    knowledge=ironwood_knowledge,
    instructions=[
        "Answer using the retrieved Ironwood documents.",
        "Quote the specific document when citing a fact (e.g. 'per team_faq.md').",
        "If the documents do not contain the answer, say so plainly. Do NOT guess.",
    ],

    
    search_knowledge=True,   # let the agent decide when to search
    markdown=True,
    debug_mode=True,

)

DEBUG Function: web_search registered with websearch

DEBUG Function: search_news registered with websearch

In [31]:
omnipotens.print_response('Who is the head of the thermal systems division?', stream=True)

DEBUG Session ID: 354f078e-a994-452b-b3a8-a778854b4a3a

DEBUG Agent initialized: librarian

DEBUG Creating new AgentSession: 354f078e-a994-452b-b3a8-a778854b4a3a

DEBUG Agent Run Start: 96d0f065-94a4-4aa9-a212-1347b14295c5

DEBUG Processing tools for model

DEBUG Added tool count_employees

... (8 outputs left out)

DEBUG =========================== user ===========================

DEBUG Who is the head of the thermal systems division?

ERROR    Error in Agent run: [Errno 111] Connection refused

DEBUG Added RunOutput to Agent Session

In [32]:
omnipotens.print_response('What is the latest way the US president has increased inflation?', stream=True)

DEBUG Session ID: 354f078e-a994-452b-b3a8-a778854b4a3a

DEBUG Agent initialized: librarian

DEBUG Creating new AgentSession: 354f078e-a994-452b-b3a8-a778854b4a3a

DEBUG Agent Run Start: fa92701f-c873-426a-bc7c-558d9d6abf1e

DEBUG Processing tools for model

DEBUG Added tool count_employees

... (8 outputs left out)

DEBUG =========================== user ===========================

DEBUG What is the latest way the US president has increased inflation?

ERROR    Error in Agent run: [Errno 111] Connection refused

DEBUG Added RunOutput to Agent Session

In [33]:
omnipotens.print_response('How many emplyees are in Ironwood labs?', stream=True)

DEBUG Session ID: 354f078e-a994-452b-b3a8-a778854b4a3a

DEBUG Agent initialized: librarian

DEBUG Creating new AgentSession: 354f078e-a994-452b-b3a8-a778854b4a3a

DEBUG Agent Run Start: f2c2c8d4-cdf0-4aae-bbfd-693c6c1819a4

DEBUG Processing tools for model

DEBUG Added tool count_employees

... (8 outputs left out)

DEBUG =========================== user ===========================

DEBUG How many emplyees are in Ironwood labs?

ERROR    Error in Agent run: [Errno 111] Connection refused

DEBUG Added RunOutput to Agent Session

In [34]:
# 💻 Q4-5: stronger model and GPU comparison
# e.g. !ollama pull qwen3:8b  then rebuild the agent with Ollama(id="qwen3:8b")
# Tip: put %%time on the first line of a cell to time it


from agno.tools.websearch import WebSearchTools

!ollama pull qwen3:8b

omnipotens = Agent(
    name="Librarian",
    role="Answer questions using Ironwood Labs' internal documentation",
    
    model=Ollama(id="qwen3:8b"),
    tools=[count_employees, employees_per_division, WebSearchTools(backend="duckduckgo")],

    knowledge=ironwood_knowledge,
    instructions=[
        "Answer using the retrieved Ironwood documents.",
        "Quote the specific document when citing a fact (e.g. 'per team_faq.md').",
        "If the documents do not contain the answer, say so plainly. Do NOT guess.",
    ],

    
    search_knowledge=True,   # let the agent decide when to search
    markdown=True,
    debug_mode=True,

)

/usr/lib/python3.13/pty.py:95: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid, fd = os.forkpty()


]11;?\Error: could not connect to ollama server, run 'ollama serve' to start it


DEBUG Function: web_search registered with websearch

DEBUG Function: search_news registered with websearch

In [35]:
omnipotens.print_response('What is the latest way the US president has increased inflation?', stream=True)

DEBUG Session ID: edd53cba-935c-4ced-9f98-5dca520124db

DEBUG Agent initialized: librarian

DEBUG Creating new AgentSession: edd53cba-935c-4ced-9f98-5dca520124db

DEBUG Agent Run Start: 9263544f-3f38-4361-9921-1ebb995b6554

DEBUG Processing tools for model

DEBUG Added tool count_employees

... (8 outputs left out)

DEBUG =========================== user ===========================

DEBUG What is the latest way the US president has increased inflation?

ERROR    Error in Agent run: [Errno 111] Connection refused

DEBUG Added RunOutput to Agent Session